# What's the farthest you can go in one turn in Monopoly?

## Just thinking it out

Intuitively, you can draw 'Advance to Go' from Chance or Community Chest (CC) right off of Go then accumulate 38 squares as you loop around the board back to Go. It feels like does so much more for your distance than rolling a measley 12. So lets look at those possibilities. 

The Chance cards might get tweaked in some editions? It might mainly be for flavour, like removing references to drunk driving. It's hard to say. These are the cards I have in my 2013 copy of the game at home. Looking online, [this extremely well-researched-looking website](https://www.zpag.net/Jeux/list_of_all_monopoly_Chance_Community_Chest.html) seems to also have the same set. But here's how far you can go from the first Chance/CC squares on the board.

|Card                          | Spaces travelled |
|:---                          |---:|
| Advance to Go (from CC)      | 38|
| Advance to Go (from Chance)  | 33|
| Advance to King's Cross      | 38|
| Advance to Pall Mall         | 4 |
| Advance to Mayfair           | 32|
| Advance to nearest station   | 8|
| Advance to nearest utilities | 5|

So lets aim for three of the four biggest ones of Advance to Go, King's Cross, and Mayfair. The other thing to think about is your rolls need to be even (doubles), even (doubles), odd (no doubles to avoid jail). This means the last turn could be either: reaching Chance from Go (7 spaces), or reaching CC from Mayfair (3 spaces). There's only one way to get to Mayfair, so lets start with that.

* Your last turn is rolling a 3 from Mayfair to land on CC.
* Your previous turn was drawing a Advance to Mayfair from Chance, after rolling an even number.
  * This means you landed on an even numbered space, like King's Cross, 2 away.
* You can get to KingsX from Chance, so you landed there on the previous turn.
  * Again, that took an even number, but it's the first roll so let's put you as far away as you can be: double-sixes.
* So you started on Liverpool Street.

Putting that all together:

|Space | Roll | Card | Distance |
|:---   |:---  |:---  |---:      |
| Liverpool St| Double-6s |         | 12 |
| Chance      |           | KingsX  | 38 |
| KingsX      | Double 1s |         | 2 |
| Chance      |           | Mayfair | 32 |
| Mayfair     | 3         |         | 3 |
| CC          |           | Go      | 38 |

For a total distance of 125. That seems pretty good, but let's try something comprehensive and see how close this was.

## Computer go brrr

I'll write some Python code to just brute force it.

### Setting it all up

First we're going to define the spaces on the board. There are 40 squares in monopoly. Squares 8, 23 and 37 are Chance sqaures (Python starts counting from zero, so these are all 1 smaller in the code). Community Chest (CC) are on 3, 18, 34. Defining the stations and utilities too.

Next we're defining a set of functions that will either move the player a specific number of spaces ahead, to a particular square, or to the next station or utility.

In [ ]:
import time

BOARD_SIZE = 40
CHANCE_POSITIONS = {7, 22, 36}
CC_POSITIONS = {2, 17, 33}
STATION_POSITIONS = [5, 15, 25, 35]
UTILITIES_POSITIONS = [12, 28]

def advance_spaces(pos, spaces):
    new_pos = (pos + spaces) % BOARD_SIZE
    distance = spaces
    return new_pos, distance

def advance_to(pos, target):
    new_pos = target
    distance = (target - pos) % BOARD_SIZE
    return new_pos, distance

def nearest_station(pos):
    for s in STATION_POSITIONS:
        if s > pos:
            return s
    return STATION_POSITIONS[0]

def nearest_utility(pos):
    for u in UTILITIES_POSITIONS:
        if u > pos:
            return u
    return UTILITIES_POSITIONS[0]


Next we define the Chance and CC cards. I'm only bothering with the ones that advance the player any number of spaces. So I label them, then define a function that will move the player to the right square for each of the labels. Same for the CC cards.

In [ ]:
CHANCE_CARDS = {
    'go',           # Advance to Go
    'kingsx',       # Advance to King's Cross Station (5)
    'pall_mall',    # Advance to Pall Mall (11)
    'trafalgar',    # Advance to Trafalgar Square (24)
    'mayfair',      # Advance to Mayfair (39)
    'utility',      # Advance to nearest utility
    'station1',     # Advance to nearest station
    'station2',     # Advance to nearest station
    'back3',        # Go back 3
    'jail',         # Go to jail, end turn
}

def apply_chance_card(pos, card):
    if card == 'go': return advance_to(pos, 0)
    if card == 'kingsx': return advance_to(pos, 5)
    if card == 'pall_mall': return advance_to(pos, 11)
    if card == 'trafalgar': return advance_to(pos, 24)
    if card == 'mayfair': return advance_to(pos, 39)
    if card == 'utility': return advance_to(pos, nearest_utility(pos))
    if card == 'station1' or card == 'station2': return advance_to(pos, nearest_station(pos))
    if card == 'back3': return advance_spaces(pos, -3)
    if card == 'jail': return advance_spaces(pos, 0) # Use zero spaces as proxy for ending turn

CC_CARDS = {'go', 'jail'}
def apply_cc_card(pos, card):
    if card == 'go': return advance_to(pos, 0)
    if card == 'jail': return advance_spaces(pos,0)


We're only interested in the total value of the rolls and whether or not it's a doubles roll. So I define a list of all the possible combinations of total values with doubles True/False.


In [ ]:
ROLLS = [
    (2, True),
                (3, False),
    (4, True),  (4, False),
                (5, False),
    (6, True),  (6, False),
                (7, False),
    (8, True),  (8, False),
                (9, False),
    (10, True), (10, False),
                (11, False),
    (12, True),
]



Next the first big function. When landing on a square we need to check if you draw any cards. First, log the square you land on. This captures squares that aren't cards spaces, but also captures what happens if the Chance or CC card you draw doesn't move you.

Then, if you landed on a Chance square, cycle through every possible Chance card that moves the player and store that new position. Again, I do this immediately to capture the case where you draw a turn-ending card in from CC space 33 if you end up there. Becuase: it's possible to draw the 'Go back three spaces' on Chance space 36, and end up on CC space 33, where I make one last to check to see if the 'Go' card is still in the deck, draw it, advance to go, then log that outcome.

In [ ]:

def resolve_landing(pos, dist, chance_used, cc_used):
    # Add an outcome if no movement card is available
    outcomes = [(pos, dist, chance_used, cc_used)]
    if pos in CHANCE_POSITIONS:
        for chance_card in CHANCE_CARDS:  
            if chance_card not in chance_used:
                # If you land on chance, loop through the unused cards. Apply them, add them to the list of used cards
                new_pos, new_dist = apply_chance_card(pos, chance_card) 
                dist_total = dist + new_dist
                updated_chance_used = chance_used | {chance_card}
                
                # Log where you land, this will cover the case where Go has been drawn already
                outcomes.append((new_pos, dist_total, updated_chance_used, cc_used))
                if new_pos in CC_POSITIONS:
                    for cc_card in CC_CARDS:
                        if cc_card not in cc_used:
                            new_pos, new_dist = apply_cc_card(new_pos, cc_card)
                            dist_total += new_dist
                            updated_cc_used = cc_used | {cc_card}
                            outcomes.append((new_pos, dist_total, updated_chance_used, updated_cc_used))
    
    elif pos in CC_POSITIONS:
        # Loop through possible CC cards
        for cc_card in CC_CARDS:
            if cc_card not in cc_used:
                new_pos, new_dist = apply_cc_card(pos, cc_card)
                dist_total = dist + new_dist
                updated_cc_used = cc_used | {cc_card}
                outcomes.append((new_pos, dist_total, chance_used, updated_cc_used))
    return outcomes


The other big function. This runs a single roll, then, if you rolled doubles, and didn't go to jail for speeding or by drawing a card, it will call itself again. As it runs through, it writes down the route you've taken through the board. Every time it calls resolve_landing() and reaches a dead end it goes through each of the outcomes and checks how far you've moved. If it's a new record, it stores the path that got you there.

In [ ]:
def search(pos, dist_total, chance_used, cc_used, path, rolls_taken, max_rolls):
    global best_distance, best_info, checked_turns
    rolls_taken += 1
    
    for roll, is_doubles in ROLLS:
        if rolls_taken == max_rolls and is_doubles:
            continue # This is the third doubles roll. Go straight to jail, don't evaluate the roll

        new_pos, dist_roll = advance_spaces(pos, roll)
        states = resolve_landing(new_pos, dist_total + dist_roll, chance_used, cc_used)

        for state in states:
            # Keep a log of the roll, square and any drawn cards
            new_path = path + [(f"Roll[{rolls_taken}]",pos, roll, is_doubles, new_pos, state[2] - chance_used, state[3] - cc_used)]

            if not is_doubles or 'jail' in state[2] or 'jail' in state[3]:
                # Without a doubles roll, or if you drew a jail card, the turn ends, check if it's the longest path
                checked_turns += 1
                if state[1] > best_distance:
                    best_distance = state[1]
                    best_info = [(best_distance, *new_path)]
                elif state[1] == best_distance:
                    best_info.append((best_distance, *new_path))
            else:
                search(*state, new_path, rolls_taken, max_rolls)                        


### Results

Here we actually run the thing. Set up a bunch of global variables and run the search() function for each of the 40 possible starting squares. I also check how long this takes, about 0.1s for me.

In [ ]:
start = time.perf_counter()
max_rolls = 3
best_distance = 0
best_info = []
checked_turns = 0
for starting_square in range(BOARD_SIZE):
    search(starting_square, 0, set(), set(), [], 0, max_rolls)

print(f"Elapsed: {time.perf_counter() - start:.3f}s")


Elapsed: 0.116s


In [ ]:
checked_turns

85038

I was curious to see how many possible turns there are. 85,000! So, lots, but totally a computable space.

In [ ]:
best_info

[(130,
  ('Roll[1]', 35, 12, True, 7, {'go'}, set()),
  ('Roll[2]', 0, 2, True, 2, set(), {'go'}),
  ('Roll[3]', 0, 7, False, 7, {'kingsx'}, set())),
 (130,
  ('Roll[1]', 35, 12, True, 7, {'mayfair'}, set()),
  ('Roll[2]', 39, 8, True, 7, {'go'}, set()),
  ('Roll[3]', 0, 7, False, 7, {'kingsx'}, set()))]

And this reports the best turn. I find **the farthest you can go is 130 squares**. There's two ways do do that: 
* They both start on Liverpool Street Station (square 36)
* Then roll double-sixes, draw a card from Chance on square 8 to move you either to Mayfair or Go
* From there get your way to the Community Chest on square 3 or the same Chance square on 8 by rolling doubles
* Draw the 'Advance to Go' card on whichever space you landed on
* Finally, roll any 7 to get to the same Chance square and go to King's Cross Station on square 6.

This is just a bit farther than my pen-and-paper solution up top. And I was close!

Where did I go off track? When working backwards, I decided to consider only where you reach CC from Mayfair since that only had one option to evaluate. But if I'd treid to handle the two options for landing on Chance as the final move, that would have set me on a path to the best answer though!

But it's nice to see the real answer lines up pretty well with our intuition for how to find it. There's not a lot of funny business in to get really far in Monopoly, you just have to draw the right cards.

Above, this is printing the best routes it found. The way to read these outputs is:
```
[(Total distance,
  (Which roll, Starting square, Roll value, Doubles?, Landing square, Chance card drawn, CC card drawn), 
  etc...)
```

Finally, what are the odds of having a perfect turn in Monopoly? Fortunately, it's the product of some independent outcomes so we can just do some arithmetic. Important to know there are 16 cards in each of the Chance and CC decks, and there's only one of each of the relevant cards. And for simplicity's sake we'll assume you're equally likely to start your turn on any square, a 1-in-40 chance. But you might be marginally more likely to find yourself on a station like Liverpool Street.

|Route 1 | Odds | Route 2| Odds |
|:---    | ---:|:--- | ---:|
| Start on L'pool St | 1/40                   | Start on L'pool St |1/40 |
| Roll double 6 | 1/36                        | Roll double 6 | 1/36 |
| Land on Chance, draw Go | 1/16              | Land on Chance, draw Mayfair | 1/16 |
| Land on Go, roll double 2 | 1/36            | Land on Mayfair, roll double 4 | 1/36 |
| Land on CC, draw Go | 1/16                  | Land on Chance, draw Go | 1/15 |
| Land on Go, roll 7 | 6/36                   | Land on Go, roll 7 | 6/36 |
| Land on Chance, draw KX | 1/15              | Land on Chance, draw KX | 1/14 |
| Total odds | 1/1,194,393,600                | Total odds | 1/1,045,094,400 |

Adding those total odds together we get **1 in 557,383,680**.

What does that mean? [Guiness says](https://www.guinnessworldrecords.com/world-records/72473-most-popular-board-game) in 1999, Monopoly had been played by 500,000,000 people worldwide. I reckon I've played, maybe four games of monopoly in my life? I remember one big fight, and one Disney-branded version, and a game I played over the Christmas holidays we abandoned after serveral hours. So lets say, as of 1999, there had been 2 billion person-games of Monopoly. The very smart people over at [Ted-Ed blogs](https://blog.ed.ted.com/2017/12/01/heres-how-to-win-at-monopoly-according-to-math-experts/) guess that "the average game of Monopoly takes about 30 turns per competitor". Now we're getting somewhere. That would suggest 60 billion turns of Monopoly have been played the world over in 1999. The odds that *nobody* has ever done a perfect turn are a mere (1-1/557,383,680)<sup>6×10^10</sup>, which is some fraction approaching zero.